# 03 — Feature Engineering & Preprocessing Pipelines
## EMIPredict AI Platform
This notebook demonstrates domain-specific financial ratio calculations, Scikit-Learn transformers, and ethical AI fairness exclusions.


In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.data.prepare_data import prepare_sample_and_splits
from src.features.build_features import calculate_financial_features
from src.features.preprocessing import create_preprocessor, get_model_feature_lists


### 1. Partition Data (70% Train, 15% Val, 15% Test)
Strict rule: preprocessing pipelines are fitted exclusively on training data to prevent leakage.


In [ ]:
train_df, val_df, test_df, sample_df = prepare_sample_and_splits()
print(f'Train partition: {len(train_df):,} rows')
print(f'Validation partition: {len(val_df):,} rows')
print(f'Test partition: {len(test_df):,} rows')


### 2. Compute Engineered Financial Ratios
We calculate 13 domain metrics including Current DTI, Expense Ratio, Obligation Ratio, Proposed Principal Burden, Disposable Income, Emergency Runway, and Savings Ratio with division-by-zero safeguards.


In [ ]:
train_features = calculate_financial_features(train_df)
train_features[['monthly_salary', 'current_debt_to_income_ratio', 'expense_to_income_ratio', 'proposed_principal_burden_ratio', 'disposable_income', 'emergency_fund_months']].head()


### 3. Fit ColumnTransformer Preprocessing Pipeline
We verify that sensitive attributes (`gender`, `marital_status`) are excluded from model features, and numeric/categorical transformers are assembled.


In [ ]:
num_cols, cat_cols, sensitive = get_model_feature_lists()
print(f'Numerical features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Excluded sensitive attributes for fairness: {sensitive}')

preprocessor = create_preprocessor(include_feature_engineering=True)
X_train = train_df.drop(columns=['emi_eligibility', 'max_monthly_emi'])
X_transformed = preprocessor.fit_transform(X_train)
print(f'Transformed feature matrix shape: {X_transformed.shape}')


### Summary
The feature engineering and preprocessing pipeline transforms raw applicant data into a robust, leak-free, scaled matrix ready for classification and regression modeling.
